# PyTorch Lightning fraud classifier

Run this in a separate fresh Colab runtime from Keras to avoid package conflicts. Record the actual Flash install attempt and the route used; do not claim an install result before performing the attempt.

## Execution provenance

All code cells below were executed on 16 November 2026 in an isolated CPU checkout. These are fresh verification-training outputs, recorded in `results/verification/2026-11-16/lightning.json`. The published report and slides use the earlier frozen `results/lightning.json`; the rerun did not overwrite that experiment. Different software, hardware and training randomness can change the metrics. See `docs/demo_readiness.md` for the comparison. Run on CPU with GPUs hidden (CUDA_VISIBLE_DEVICES=-1).


## Setup and shared data

In [1]:
from pathlib import Path
import os, subprocess
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (repo_root / "src" / "data.py").exists():
    repo_root = Path("/content/credit-card-fraud-detection")
    if not (repo_root / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git", str(repo_root)], check=True)
os.chdir(repo_root)
print("Project root:", Path.cwd())


Project root: /home/vanshsharma/ccfd-rerun/train


In [2]:
%pip install -q -r requirements/base.txt
%pip install -q -r requirements/lightning.txt
import subprocess, sys
subprocess.run([sys.executable, "src/data.py"], check=True)
# In the full team repository, run the existing setup/data cells first.
# The team's src/data.py must expose load_splits() as agreed in the project plan.
from pathlib import Path
import sys, random, time, platform
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() / "src"))
from data import load_splits
from evaluate import evaluate_model, permutation_importance
SEED = 42
random.seed(SEED); np.random.seed(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torchmetrics.classification import BinaryAveragePrecision
L.seed_everything(42, workers=True)

(X_train, y_train, X_val, y_val, X_test, y_test,
 row_id_val, row_id_test, amount_val, amount_test) = load_splits()
feature_names = list(X_train.columns) if hasattr(X_train, "columns") else [f"x{i}" for i in range(X_train.shape[1])]
X_train = np.asarray(X_train, dtype=np.float32); X_val = np.asarray(X_val, dtype=np.float32); X_test = np.asarray(X_test, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int64).reshape(-1); y_val = np.asarray(y_val, dtype=np.int64).reshape(-1); y_test = np.asarray(y_test, dtype=np.int64).reshape(-1)
fraud_weight = float((y_train == 0).sum() / (y_train == 1).sum())


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Raw rows: 284807, raw frauds: 492
Duplicates dropped: 1081
Rows after cleaning: 283726, frauds: 473
train rows:  170235  frauds:  284  fraud share: 0.1668%
val   rows:   56745  frauds:   94  fraud share: 0.1657%
test  rows:   56746  frauds:   95  fraud share: 0.1674%
Class weight (train normal / train fraud): 169951 / 284 = 598.4190


Seed set to 42


In [3]:
class FraudNet(L.LightningModule):
    def __init__(self, n_features, pos_weight):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1),
        )
        self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float32))
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        self.val_ap = BinaryAveragePrecision()

    def forward(self, x): return self.net(x).squeeze(1)
    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y.float())
        self.log("train_loss", loss, on_epoch=True, prog_bar=False)
        return loss
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y.float())
        self.val_ap.update(torch.sigmoid(logits), y)
        self.log("val_loss", loss, on_step=False, on_epoch=True)
        self.log("val_pr_auc", self.val_ap, on_step=False, on_epoch=True, prog_bar=True)
    def configure_optimizers(self): return torch.optim.Adam(self.parameters(), lr=0.001)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)


/tmp/ipykernel_66771/3074220027.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:208.)
  train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))


## Train and evaluate

In [4]:
model = FraudNet(X_train.shape[1], fraud_weight)
Path("models").mkdir(exist_ok=True)
checkpoint = ModelCheckpoint(dirpath="models/lightning_checkpoints", monitor="val_pr_auc", mode="max", save_top_k=1, filename="best-{epoch:02d}-{val_pr_auc:.4f}")
stop = EarlyStopping(monitor="val_pr_auc", mode="max", patience=5)
trainer = L.Trainer(max_epochs=30, accelerator="auto", devices=1, callbacks=[checkpoint, stop], log_every_n_steps=20, enable_progress_bar=True)
start = time.perf_counter()
trainer.fit(model, train_loader, val_loader)
train_seconds = time.perf_counter() - start
best_model = FraudNet.load_from_checkpoint(checkpoint.best_model_path, n_features=X_train.shape[1], pos_weight=fraud_weight)
best_model.eval(); device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); best_model.to(device)
def predict_probabilities(X):
    loader = DataLoader(torch.from_numpy(np.asarray(X, dtype=np.float32)), batch_size=8192, shuffle=False)
    values=[]
    with torch.no_grad():
        for xb in loader: values.append(torch.sigmoid(best_model(xb.to(device))).cpu().numpy())
    return np.concatenate(values)
p_val = predict_probabilities(X_val)
start = time.perf_counter()
p_test = predict_probabilities(X_test)
predict_seconds = time.perf_counter() - start
import shutil
shutil.copy2(checkpoint.best_model_path, "models/lightning.ckpt")
result = evaluate_model("lightning", y_val, p_val, y_test, p_test, row_id_val, row_id_test,
    train_seconds, predict_seconds,
    notes="Plain PyTorch Lightning used; Flash fallback: recorded install failure in docs/person2_results.md. Shared 64-32 network; class-weighted BCE; early stopping on validation PR-AUC.",
    hardware=f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
result


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


/home/vanshsharma/ccfd-rerun/venv-torch/lib64/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net     │ Sequential             │  4.1 K │ train │     0 │
│ 1 │ loss_fn │ BCEWithLogitsLoss      │      0 │ train │     0 │
│ 2 │ val_ap  │ BinaryAveragePrecision │      0 │ train │     0 │
└───┴─────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.1 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.1 K                                                                                                
Total estimated model params size (MB): 0.016                                                                      
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/vanshsharma/ccfd-rerun/venv-torch/lib64/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/vanshsharma/ccfd-rerun/venv-torch/lib64/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/vanshsharma/ccfd-rerun/venv-torch/lib64/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_co
nnector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/home/vanshsharma/ccfd-rerun/venv-torch/lib64/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_co
nnector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

{'framework': 'lightning',
 'threshold': 0.9999673366546631,
 'precision': 0.8837209302325582,
 'recall': 0.8,
 'f1': 0.8397790055248618,
 'pr_auc': 0.7581131041239888,
 'roc_auc': 0.9626652198270296,
 'tn': 56641,
 'fp': 10,
 'fn': 19,
 'tp': 76,
 'rmse': 0.1241529425985869,
 'mae': 0.03761293268658104,
 'r2': -8.222599651699385,
 'train_seconds': 32.82318933000033,
 'predict_seconds': 0.08248172399999021,
 'hardware': 'CPU',
 'notes': 'Plain PyTorch Lightning used; Flash fallback: recorded install failure in docs/person2_results.md. Shared 64-32 network; class-weighted BCE; early stopping on validation PR-AUC.'}

## Validation permutation importance

In [5]:
def lightning_predict(frame):
    return predict_probabilities(np.asarray(frame, dtype=np.float32))
perm = permutation_importance(lightning_predict, pd.DataFrame(X_val, columns=feature_names), y_val, n_repeats=3, random_state=42)
Path("results").mkdir(exist_ok=True)
perm.to_csv("results/perm_lightning.csv", index=False)
perm.head(10)

,feature,importance_mean,importance_std,baseline_pr_auc
0,V14,0.061189,0.000686,0.709535
1,V12,0.031525,0.000320,0.709535
2,V4,0.029940,0.000418,0.709535
3,V17,0.016194,0.001328,0.709535
4,V10,0.012000,0.001207,0.709535
5,V11,0.009681,0.004741,0.709535
6,V9,0.008130,0.001799,0.709535
7,V3,0.007899,0.000681,0.709535
8,V16,0.006907,0.002929,0.709535
9,V8,0.005278,0.000918,0.709535
